This will be a notebook used for prototyping my Neo4j powered recommendation ML algorithm (FPG). This is the foundation that powers all recommendation systems

* FPG Algorithm -- https://www.geeksforgeeks.org/machine-learning/frequent-pattern-growth-algorithm/

###

In [ ]:
def install_dependencies(libraries):
  for library in libraries:
    !pip install "{library}"

In [ ]:
%%capture

dependencies = [
    "neo4j ",
    "python-dotenv",
]

install_dependencies(dependencies)

In [ ]:
import dotenv
dotenv.load_dotenv()

True

###Test Transactions

In [ ]:
#test transactions
transactions = [
    ['milk', 'butter', 'bread'],
    ['bread', 'butter'],
    ['bread', 'milk'],
    ['milk', 'butter'],
    ['bread']
]

###Step 1: Compute Item Frequencies

In [ ]:
from collections import Counter

items_counter = Counter()
for transaction in transactions:
  items_counter.update(transaction)

items_counter

Counter({'milk': 3, 'butter': 3, 'bread': 4})

###Step 2: Order Items in Each Transaction by Frequency

Also, removing the items that are below the minimum amount of support.

In [ ]:
#finds the rightmost index with the minimum amount of support
def filter_transaction(transaction, items_counter, min_support=2):

  if len(transaction) == 0:
    return None

  if items_counter[transaction[0]] < min_support:
    return None

  if items_counter[transaction[-1]] >= min_support:
    return transaction

  left = 0
  right = len(transaction) - 1

  while left < right:
    mid = (left + right) // 2
    if items_counter[transaction[mid]] >= min_support:
      left = mid + 1
    else:
      right = mid

  return transaction[:left]

In [ ]:
sorted_transactions = []
for transaction in transactions:
  sample_list = sorted(transaction, key=lambda item: items_counter[item], reverse=True)
  sample_list = filter_transaction(sample_list, items_counter, min_support=2)

  if sample_list is not None:
    sorted_transactions.append(sample_list)

sorted_transactions

[['bread', 'milk', 'butter'],
 ['bread', 'butter'],
 ['bread', 'milk'],
 ['milk', 'butter'],
 ['bread']]

###Step 3: Construct FP-Tree

Neo4j is the perfect database for this because of its graphical capabilities.

Documentation: https://neo4j.com/docs/python-manual/current/query-simple/

In [ ]:
#the nodes that are the
class Node:
  def __init__(self, name, parent=None):
    self.name = name
    self.children = {}
    self.count = 0
    self.parent = parent

In [ ]:
from collections import deque

class FP_Tree:
  def __init__(self):
    self.root = None

  """
  Constructs the FP tree from transaction information
  """
  def construct_tree(self, transactions):
    self.root = Node('root')

    for transaction in sorted_transactions:
      current = self.root
      for item in transaction:
        if item not in current.children:
          current.children[item] = Node(item, parent=current)

        current = current.children[item]
        current.count += 1

    return self.root

  """
  Prints the FP tree
  """
  def print_tree(self):
    queue = deque()
    queue.append(self.root)

    while len(queue) > 0:
      for i in range(len(queue)):
        current = queue.popleft()
        parent_name = ""

        if current.parent is not None:
          parent_name = current.parent.name

        print(f"{current.name}, {current.count}, {parent_name}")

        for child in current.children.values():
          queue.append(child)

      print('-'*30)

In [ ]:
tree = FP_Tree()
root = tree.construct_tree(sorted_transactions)

In [ ]:
tree.print_tree()

root, 0, 
------------------------------
bread, 4, root
milk, 1, root
------------------------------
milk, 2, bread
butter, 1, bread
butter, 1, milk
------------------------------
butter, 1, milk
------------------------------


###Step 4: Determine Conditional Pattern Bases

In [ ]:
"""
Function that constructs the conditional pattern base for a given node.
params: node (a node of the tree)
path: list[node] (list of the node path)
pattern_base: dict[list[node]] (dictionary of pattern bases)
"""
def construct_conditional_pattern_base(node, path, pattern_base):
  pattern_base[node.name].append((set(path), node.count))

  for child_node in node.children.values():
    construct_conditional_pattern_base(child_node, path+[node.name], pattern_base)

In [ ]:
root_children = root.children
pattern_base = {item:[] for item in items_counter}

for child in root_children.values():
  construct_conditional_pattern_base(child, [], pattern_base)

In [ ]:
pattern_base

{'milk': [({'bread'}, 2), (set(), 1)],
 'butter': [({'bread', 'milk'}, 1), ({'bread'}, 1), ({'milk'}, 1)],
 'bread': [(set(), 4)]}

###Step 5: Build Conditional FP-Trees

In [ ]:
import copy

"""
Function that builds conditional fp trees.
params: pattern_base (dictionary of pattern bases)
min_support (minimum amount of support)
"""
def build_conditional_fp_tree(pattern_base, min_support=2):
  final_transactions = set()

  #generate a counter of all items in pattern_base
  for key, tup in pattern_base.items():
    item_counter = Counter()

    for path in tup:
      if(len(path[0]) > 0):
        item_count = {item: path[1] for item in path[0]}
        item_counter.update(item_count)

    #generate a set of all elements less than the min_support
    min_set = set()

    for item, count in item_counter.items():
      if count < min_support:
        min_set.add(item)

    #take the difference of sets, make them frozensets, and store them
    for path in tup:
      # Create a new set that is the union of path[0] and {key}
      full_set = path[0].union({key})
      sample_set = frozenset(full_set.difference(min_set))
      final_transactions.add(sample_set)

  return final_transactions

In [ ]:
final_transactions = build_conditional_fp_tree(pattern_base)

for item in items_counter:
  final_transactions.add(frozenset({item}))

In [ ]:
final_transactions

{frozenset({'butter'}),
 frozenset({'milk'}),
 frozenset({'bread'}),
 frozenset({'bread', 'butter'}),
 frozenset({'bread', 'milk'}),
 frozenset({'butter', 'milk'}),
 frozenset({'bread', 'butter', 'milk'})}

## Graphical Representation in Neo4j

We'll use the `neo4j` Python driver to connect to your Neo4j database and load the frequent itemsets. Each unique item will be a node labeled `Item`, and we'll create relationships to represent their co-occurrence within the discovered frequent itemsets. For simplicity, we can create a `Frequents` relationship between items in the same itemset.

Make sure your Neo4j instance is running and you have the connection details (URI, username, password).

In [ ]:
final_transactions

{frozenset({'butter'}),
 frozenset({'milk'}),
 frozenset({'bread'}),
 frozenset({'bread', 'butter'}),
 frozenset({'bread', 'milk'}),
 frozenset({'butter', 'milk'}),
 frozenset({'bread', 'butter', 'milk'})}

In [ ]:
from neo4j import GraphDatabase
import os

# Load Neo4j credentials from environment variables
NEO4J_URI = os.getenv('NEO_CONNECTION_URL')
NEO4J_USERNAME = os.getenv('NEO_DB_ID')
NEO4J_PASSWORD = os.getenv('NEO_PASSWORD')

# Ensure credentials are set
if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError("Neo4j credentials (NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD) must be set in environment variables or Colab secrets.")

# Establish connection to Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

print("Connected to Neo4j!")

Connected to Neo4j!


In [ ]:
with driver.session() as session:
    # Using session.run() for administrative commands
    session.run("MATCH (n) DETACH DELETE n", db="recommendations")

In [ ]:
def write_frequent_itemsets_to_neo4j(tx, itemsets):
    # Create nodes for all items and relationships for each itemset
    for itemset in itemsets:
        item_names = list(itemset)
        if len(item_names) == 1:
            # For single items, just create the node
            item_name = item_names[0]
            tx.run("MERGE (i:Item {name: $item_name});", item_name=item_name)
        else:
            # For itemsets with multiple items, create nodes and connect them efficiently
            tx.run(
                "UNWIND $item_names AS item1_name \n"
                "UNWIND $item_names AS item2_name \n"
                "WITH item1_name, item2_name \n"
                "WHERE item1_name < item2_name // Ensure unique pairs and avoid self-loops \n"
                "MERGE (i1:Item {name: item1_name}) \n"
                "MERGE (i2:Item {name: item2_name}) \n"
                "MERGE (i1)-[:ASSOCIATE]-(i2)",
                item_names=item_names
            )

# Write the frequent itemsets to the database using the optimized function
with driver.session() as session:
    # Ensure unique constraints for Item nodes (this must be outside the write_transaction or in its own transaction)
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.name IS UNIQUE;")

    session.execute_write(write_frequent_itemsets_to_neo4j, final_transactions)
    print(f"Successfully wrote {len(final_transactions)} frequent itemsets to Neo4j.")

# Close the driver connection
driver.close()
print("Neo4j connection closed.")

Successfully wrote 7 frequent itemsets to Neo4j.
Neo4j connection closed.
